# Phase 1: Single Session Explorer

Interactive dashboard for exploring kinematics data from a single session.

**Features:**
- Load any session from the registry
- Select perturbation direction and trial type
- Adjust visualization time windows
- View aligned kinematics (18 subplots) and 2D trajectories
- Display statistics table at perturbation onset

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128  # Remove file size limit
from ipywidgets import (
    interact, interactive, fixed, IntSlider, Dropdown, Button, HBox, VBox,
    Label, Output, FloatSlider
)
from IPython.display import display, clear_output, HTML, Image

from tools.behavior import (
    BehaviorDataset, TimeSeriesPlotter, load_session_registry
)

print("✓ Imports successful")

✓ Imports successful


## Load Session Registry

In [2]:
# Load the session registry
registry = load_session_registry()
sessions = [s.session_name for s in registry.sessions]
animals = registry.animals()
conditions = registry.conditions()

print(f"✓ Loaded {len(sessions)} sessions")
print(f"  Animals: {animals}")
print(f"  Conditions: {conditions}")

✓ Loaded 25 sessions
  Animals: ['M061', 'M062', 'M063', 'M078', 'M081', 'M086', 'M103', 'M106']
  Conditions: ['control', 'muscimol', 'normal']


## Initialize Controls

In [ ]:
# Session selection
session_dropdown = Dropdown(
    options=sessions,
    value=sessions[0],
    description='Session:',
    layout={'width': '300px'}
)

# Direction slider (0-11)
direction_slider = IntSlider(
    value=0,
    min=0,
    max=11,
    step=1,
    description='Direction:',
    layout={'width': '400px'}
)

# Trial type selection
trial_type_dropdown = Dropdown(
    options=['trial', 'free0', 'free1', 'intertrial'],
    value='trial',
    description='Trial Type:',
    layout={'width': '300px'}
)

# Time window controls
pre_ms_slider = IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description='Pre (ms):',
    layout={'width': '400px'}
)

post_ms_slider = IntSlider(
    value=1500,
    min=500,
    max=3000,
    step=100,
    description='Post (ms):',
    layout={'width': '400px'}
)

print("✓ Controls created")

✓ Controls created


In [4]:
# Trial index selector (for single trial animation)
trial_idx_slider = IntSlider(
    value=0,
    min=0,
    max=10,  # Will update based on available trials
    step=1,
    description='Trial #:',
    layout={'width': '400px'}
)

print("✓ Trial selector created")

✓ Trial selector created


## Visualization Functions

In [5]:
# Global state
current_dataset = None
current_plotter = None
current_session_name = None
plot_outputs = {}  # Store output widgets for updating

def load_session(session_name):
    """Load a session and create a plotter instance."""
    global current_dataset, current_plotter, current_session_name
    
    if session_name == current_session_name:
        return  # Already loaded
    
    print(f"Loading {session_name}...")
    current_dataset = BehaviorDataset(session_name)
    current_plotter = TimeSeriesPlotter(current_dataset)
    current_session_name = session_name
    print(f"✓ Loaded: {current_dataset}")
    
    # Update direction slider range based on available directions
    max_dir = max(current_dataset.directions) if current_dataset.directions else 11
    direction_slider.max = max_dir
    direction_slider.value = min(direction_slider.value, max_dir)  # Clamp to valid range
    print(f"  Available directions: {current_dataset.directions}")
    
    # Trigger plot updates by simulating slider change
    print(f"  Updating plots...")
    trigger_plot_updates()


def trigger_plot_updates():
    """Force all plot widgets to update with current data."""
    # Force re-execution by momentarily changing and resetting a slider
    old_val = direction_slider.value
    direction_slider.value = (old_val + 1) if old_val < direction_slider.max else old_val - 1
    direction_slider.value = old_val


def on_session_change(change):
    """Callback when session is changed."""
    load_session(change['new'])


session_dropdown.observe(on_session_change, names='value')

# Load initial session from dropdown
# load_session(session_dropdown.value)
# print("\n✓ Functions defined")

In [6]:
load_session('M103_2026_02_18_15_30')

Loading M103_2026_02_18_15_30...
M103_2026_02_18_15_30
fields: ['values_before_camera_trigger', 'idx_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
Repairing columns ['MotSen1_X', 'MotSen1_Y']
Extending index to 41999 in trial: free and id: 996, inserting NaN.
Extending index to 41999 in trial: free and id: 996, inserting NaN.
Combined every 1 bins
Resulting all_spikes ephys data shape is (NxT): (15, 48000)
Resulting CP_spikes ephys data shape is (NxT): (267, 48000)
Resulti

## Interactive Plot: Kinematics Time Series

In [ ]:
def plot_kinematics_view(direction, trial_type, pre_ms, post_ms):
    """Update kinematics plot."""
    if current_plotter is None or direction > max(current_dataset.directions):
        print(f"⚠ Direction {direction} not available")
        return
    
    try:
        fig, axes = current_plotter.plot_kinematics_grid(
            direction, trial_type, pre_ms, post_ms, show_sem=True
        )
        plt.show()
    except Exception as e:
        print(f"✗ Error: {str(e)[:100]}")

print("Creating interactive plotter...")
out_kinematics = interactive(plot_kinematics_view,
                             direction=direction_slider,
                             trial_type=trial_type_dropdown,
                             pre_ms=pre_ms_slider,
                             post_ms=post_ms_slider)

display(VBox([
    Label("Kinematics Time Series (18 subplots: 6 body parts × 3 coordinates)"),
    HBox([session_dropdown]),
    HBox([trial_type_dropdown, direction_slider]),
    HBox([pre_ms_slider]),
    HBox([post_ms_slider]),
    out_kinematics
]))

Creating interactive plotter...


## Interactive Plot: 2D Trajectories

In [ ]:
def plot_trajectories_view(direction, trial_type, pre_ms, post_ms):
    """Update 2D trajectory plot."""
    if current_plotter is None or direction > max(current_dataset.directions):
        print(f"⚠ Direction {direction} not available")
        return
    
    try:
        fig, ax = current_plotter.plot_trajectories_2d(
            direction, trial_type, pre_ms, post_ms, show_perturbation=True
        )
        plt.show()
    except Exception as e:
        print(f"✗ Error: {str(e)[:100]}")

print("Creating trajectory plotter...")
out_traj = interactive(plot_trajectories_view,
                       direction=direction_slider,
                       trial_type=trial_type_dropdown,
                       pre_ms=pre_ms_slider,
                       post_ms=post_ms_slider)

display(VBox([
    Label("2D Trajectories (X-Z plane for 6 body parts)"),
    out_traj
]))

Creating trajectory plotter...


In [ ]:
from IPython.display import HTML

# Store animation reference globally to prevent garbage collection
_current_animation = None

def plot_animation_view(direction, trial_type, pre_ms, post_ms):
    """Create and display 2D trajectory animation."""
    global _current_animation
    
    if current_plotter is None or direction > max(current_dataset.directions):
        print(f"⚠ Direction {direction} not available")
        return
    
    try:
        print(f"Generating animation for direction {direction}, {trial_type} trials...")
        fig, anim = current_plotter.plot_trajectories_animation(
            direction, trial_type, pre_ms, post_ms, fps=20
        )
        # Keep reference to animation in memory
        _current_animation = anim
        
        # Display as HTML5 video
        display(HTML(anim.to_jshtml()))
        print(f"✓ Animation ready - use the player controls to play/pause")
    except Exception as e:
        print(f"✗ Error: {str(e)[:200]}")
        import traceback
        traceback.print_exc()

print("Creating animation plotter...")
out_anim = interactive(plot_animation_view,
                       direction=direction_slider,
                       trial_type=trial_type_dropdown,
                       pre_ms=pre_ms_slider,
                       post_ms=post_ms_slider)

display(VBox([
    Label("2D Trajectory Animation (Mean) - Use player controls"),
    out_anim
]))

Creating animation plotter...


/data/miniconda3/envs/eq/lib/python3.10/site-packages/matplotlib/animation.py:908: UserWarning: Animation was deleted without rendering anything. This is most likely not intended. To prevent deletion, assign the Animation to a variable, e.g. `anim`, that exists until you output the Animation using `plt.show()` or `anim.save()`.
  warnings.warn(


## Single Trial Animation

In [13]:
# Update trial slider max based on current dataset
def update_trial_slider_range(trial_type):
    """Update trial slider range based on available trials."""
    if current_dataset is None:
        return
    max_direction = max(current_dataset.directions) if current_dataset.directions else 11
    direction = direction_slider.value
    if direction <= max_direction:
        n_trials = current_dataset.n_trials_per_direction.get(direction, 0)
        trial_idx_slider.max = max(0, n_trials - 1)
    
def on_trial_type_change(change):
    """Update trial indices when trial type changes."""
    update_trial_slider_range(change['new'])

trial_type_dropdown.observe(on_trial_type_change, names='value')

# Store animation reference for single trials
_current_single_trial_animation = None

def plot_single_trial_animation_view(direction, trial_idx, trial_type, pre_ms, post_ms):
    """Create and display 2D trajectory animation for a single trial."""
    global _current_single_trial_animation
    
    if current_plotter is None or direction > max(current_dataset.directions):
        print(f"⚠ Direction {direction} not available")
        return
    
    # Check if trial index is valid
    n_trials = current_dataset.n_trials_per_direction.get(direction, 0)
    if trial_idx >= n_trials:
        print(f"⚠ Trial {trial_idx} out of range (0-{n_trials-1})")
        return
    
    try:
        print(f"Generating single-trial animation for direction {direction}, trial {trial_idx+1}/{n_trials}, {trial_type}...")
        fig, anim = current_plotter.plot_trajectories_animation_single_trial(
            direction, trial_idx, trial_type, pre_ms, post_ms, fps=20
        )
        # Keep reference to animation in memory
        _current_single_trial_animation = anim
        
        # Display as HTML5 video
        display(HTML(anim.to_jshtml()))
        print(f"✓ Animation ready - use the player controls to play/pause")
    except Exception as e:
        print(f"✗ Error: {str(e)[:200]}")
        import traceback
        traceback.print_exc()

print("Creating single-trial animation plotter...")
out_single_anim = interactive(plot_single_trial_animation_view,
                              direction=direction_slider,
                              trial_idx=trial_idx_slider,
                              trial_type=trial_type_dropdown,
                              pre_ms=pre_ms_slider,
                              post_ms=post_ms_slider)

display(VBox([
    Label("Single Trial Animation - Use player controls"),
    HBox([Label("Trial #:"), trial_idx_slider]),
    out_single_anim
]))

Creating single-trial animation plotter...


## Session Info

In [ ]:
def show_session_info():
    """Display current session information."""
    if current_dataset is None:
        print("No session loaded")
        return
    
    print(f"\n{'='*60}")
    print(f"SESSION: {current_dataset.session_name}")
    print(f"{'='*60}")
    print(f"Animal:           {current_dataset.animal_id}")
    print(f"Condition:        {current_dataset.condition}")
    print(f"\nAvailable directions: {current_dataset.directions}")
    print(f"\nTrials per direction:")
    for direction in sorted(current_dataset.directions):
        n_trials = current_dataset.n_trials_per_direction[direction]
        print(f"  Direction {direction:2d}: {n_trials:3d} trials")
    print(f"\nFree periods:")
    print(f"  Free0:     {current_dataset.free0_duration_sec:7.1f}s ({current_dataset.free0_n_frames:6d} frames)")
    print(f"  Free1:     {current_dataset.free1_duration_sec:7.1f}s ({current_dataset.free1_n_frames:6d} frames)")
    print(f"  Intertrial: {current_dataset.intertrial_duration _sec:7.1f}s ({current_dataset.intertrial_n_frames:6d} frames)")
    print(f"\nAvailable trial types: {current_dataset.available_trial_types}")
    print(f"{'='*60}\n")

show_session_info()